# Laboration 1

## data_loading_code.py

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from matplotlib import pyplot
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk import word_tokenize
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, classification_report
from torch.nn.utils.rnn import pad_sequence
from collections import defaultdict
import nltk # new
nltk.download('stopwords') # new
nltk.download('punkt_tab')
def preprocess_pandas(data, columns):
    df_ = pd.DataFrame(columns=columns)
    data['Sentence'] = data['Sentence'].str.lower()
    data['Sentence'] = data['Sentence'].replace('[a-zA-Z0-9-_.]+@[a-zA-Z0-9-_.]+', '', regex=True)                      # remove emails
    data['Sentence'] = data['Sentence'].replace('((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)    # remove IP address
    data['Sentence'] = data['Sentence'].str.replace(r'[^\w\s]','')                                                       # remove special characters
    data['Sentence'] = data['Sentence'].replace(r'\d', '', regex=True)                                                   # remove numbers
    for index, row in data.iterrows():
        word_tokens = word_tokenize(row['Sentence'])
        filtered_sent = [w for w in word_tokens if not w in stopwords.words('english')]
        df_.loc[len(df_)] = {
            "index": row['index'],
            "Class": row['Class'],
            "Sentence": " ".join(filtered_sent)
        }
    return data

# If this is the primary file that is executed (ie not an import of another file)
# if __name__ == "__main__":
# get data, pre-process and split
data = pd.read_csv("amazon_cells_labelled.txt", delimiter='\t', header=None)
data.columns = ['Sentence', 'Class']
data['index'] = data.index                                          # add new column index
columns = ['index', 'Class', 'Sentence']
data = preprocess_pandas(data, columns)                             # pre-process
training_data, validation_data, training_labels, validation_labels = train_test_split( # split the data into training, validation, and test splits
    data['Sentence'].values.astype('U'),
    data['Class'].values.astype('int32'),
    test_size=0.10,
    random_state=0,
    shuffle=True
)

# vectorize data using TFIDF and transform for PyTorch for scalability
word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=50000, max_df=0.5, use_idf=True, norm='l2')
training_data = word_vectorizer.fit_transform(training_data)        # transform texts to sparse matrix
training_data = training_data.todense()                             # convert to dense matrix for Pytorch
vocab_size = len(word_vectorizer.vocabulary_)
validation_data = word_vectorizer.transform(validation_data)
validation_data = validation_data.todense()
train_x_tensor = torch.from_numpy(np.array(training_data)).type(torch.FloatTensor)
train_y_tensor = torch.from_numpy(np.array(training_labels)).long()
validation_x_tensor = torch.from_numpy(np.array(validation_data)).type(torch.FloatTensor)
validation_y_tensor = torch.from_numpy(np.array(validation_labels)).long()

# def load_data(filepath):
#     data = pd.read_csv(filepath, delimiter='\t', header=None)
#     data.columns = ['Sentence', 'Class']
#     data['index'] = data.index
#     columns = ['index', 'Class', 'Sentence']
#     data = preprocess_pandas(data, columns)
#     training_data, validation_data, training_labels, validation_labels = train_test_split(
#         data['Sentence'].values.astype('U'), # Unicode string
#         data['Class'].values.astype(np.int32),
#         test_size=0.15,
#         shuffle=True
#     )

# # Build vocabulary
#     word_to_idx = defaultdict(lambda: 0)  # 0 will be padding
#     idx = 1
#     for sentence in training_data:
#         for word in sentence.split():
#             if word not in word_to_idx:
#                 word_to_idx[word] = idx
#                 idx += 1
#     vocab_size = len(word_to_idx) + 1  # +1 for padding index 0

#     # Convert sentences to sequences of word indices
#     def sentences_to_indices(sentences, word_to_idx):
#         seqs = []
#         for sentence in sentences:
#             seq = [word_to_idx[word] for word in sentence.split() if word in word_to_idx]
#             seqs.append(torch.tensor(seq, dtype=torch.long))
#         return pad_sequence(seqs, batch_first=True, padding_value=0)

#     train_x_tensor = sentences_to_indices(training_data, word_to_idx)
#     val_x_tensor   = sentences_to_indices(validation_data, word_to_idx)

#     train_y_tensor = torch.tensor(training_labels, dtype=torch.long)
#     val_y_tensor   = torch.tensor(validation_labels, dtype=torch.long)

#     return train_x_tensor, train_y_tensor, val_x_tensor, val_y_tensor, vocab_size





    #     word_vectorizer = TfidfVectorizer( # Text till siffror
    #     analyzer='word',
    #     ngram_range=(1,2), # rangen är mellan ett ord och tvåpar
    #     max_features=50000,
    #     max_df=0.5, # Tar bort ord som finns i över 50% av alla texter, i engelskan är det exempelvis orden: This, The, Is, a och and.
    #     use_idf=True,
    #     norm='l2' # Normaliserar vektorerna
    # )
    # training_data = word_vectorizer.fit_transform(training_data).todense()
    # validation_data = word_vectorizer.transform(validation_data).todense()
    # train_x = torch.from_numpy(np.array(training_data)).float() # Gör från NumPy array till PyTorch Tensor. Vi sparar i float [2.0, 0,3. 0,6 etc]
    # train_y = torch.from_numpy(np.array(training_labels)).long() # Vi sparar i long [0, 1, 0 etc]
    # val_x = torch.from_numpy(np.array(validation_data)).float()
    # val_y = torch.from_numpy(np.array(validation_labels)).long()
    # return train_x, train_y, val_x, val_y

<>:23: SyntaxWarning: invalid escape sequence '\.'
<>:23: SyntaxWarning: invalid escape sequence '\.'
C:\Users\david\AppData\Local\Temp\ipykernel_48104\1639201171.py:23: SyntaxWarning: invalid escape sequence '\.'
  data['Sentence'] = data['Sentence'].replace('((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)    # remove IP address
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## imports

In [2]:
import comet_ml
from comet_ml import start
from comet_ml.integration.pytorch import log_model

import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, DataLoader
from torch.utils.data import random_split

from importlib import reload
import pandas as pd
from transformers import DistilBertTokenizer, DistilBertModel

from transformers import GPT2Model
from transformers import AutoTokenizer, AutoModelForSequenceClassification

#from importlib import reload
#import data_loading_code
#reload(data_loading_code)
#from data_loading_code import *


## Load Data

Might be wise to not load the 25K file if it isn't needed then only load the regular one.

In [3]:
train_dataset_LSTM = TensorDataset(train_x_tensor, train_y_tensor)
train_loader_LSTM = DataLoader(train_dataset_LSTM, batch_size=32, shuffle=True)

val_dataset_LSTM = TensorDataset(validation_x_tensor, validation_y_tensor)
val_loader_LSTM = DataLoader(val_dataset_LSTM, batch_size=32)

In [4]:
input_size = train_x_tensor.shape[1]
print(input_size)

class LSTM(nn.Module):
    def __init__(self, hidden_size = 128):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first= True)
        self.dropout = nn.Dropout(0.15)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, 1, features)
        out, (hidden, cell) = self.lstm(x)
        return self.fc(hidden[-1])

7277


# Task 1.1
A simple neural network composed of linear layers. You may incorporate activation
functions, dropout, and other complementary layers as needed.

In [45]:
# settings
epochs = 1000 #10
lr=0.0001

# scheduler settings
patience = 3
factor = 0.1

# initialize comet
lab1_1 = comet_ml.Experiment(
    api_key="wCXnRD5xewUGxCxBYe8ePt4JY",
    workspace="kanskejoanna",
    project_name="lab1",
    name="model 1.1.1 - ANN",
    display_name="model 1.1.1 - ANN",
)

# Report multiple hyperparameters using a dictionary:
hyper_params = {
   "learning_rate": lr,
   "steps": epochs
}
lab1_1.log_parameters(hyper_params)

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kanskejoanna/lab1/af50511dc2fc4ee5be942dc554af5fee

COMET WARNING: Unknown error exporting current conda environment
COMET WARNING: Unknown error retrieving Conda package as an explicit file
COMET WARNING: Unknown error retrieving Conda information


### 1.1 ANN: Running model

In [ ]:
model_1_1 = nn.Sequential(
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Dropout(0.15),
    nn.Linear(128,2) # Negative or positive review
)

criterion = nn.CrossEntropyLoss()
#optimizer = torch.optim.SGD(model_LSTM.parameters(), lr=0.0001)
optimizer = torch.optim.Adam(model_1_1.parameters(), lr=lr)

# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#         optimizer,
#         mode="min",
#         factor = factor,
#         patience = patience # Wait x amount of epochs before reducing lr
#     )

best_val_loss = float('inf')

last_epoch_loss = 0
worse_loss_counter = 0

for epoch in range(epochs):
    model_1_1.train()

    train_running_loss = 0.0
    for batch_x, batch_y in train_loader_LSTM:

#------------------------- TRAINING -------------------------
        optimizer.zero_grad()
        output = model_1_1(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        train_running_loss+=loss.item()
    average_train_loss = train_running_loss/len(train_loader_LSTM)
    lab1_1.log_metric("model 1.1.1 - train loss", average_train_loss, step=epoch)

#------------------------- VALIDATION -------------------------
    model_1_1.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader_LSTM:
            output = model_1_1(batch_x)
            val_loss = criterion(output, batch_y)
            val_running_loss += val_loss.item()
        average_val_loss = val_running_loss/len(val_loader_LSTM)
    lab1_1.log_metric("model 1.1.1 - validation loss", average_val_loss, step=epoch)

    # scheduler.step(average_val_loss)
    lab1_1.log_metric("model 1.1.1 - learning rate", optimizer.param_groups[0]['lr'], step=epoch)

#------------------------- VISUALIZATION -------------------------
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"train loss: {average_train_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )
#------------------------- SAVING BEST MODEL -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model_1_1.state_dict(), "BestModel_1_1.pth")
        print("Saved The Best Performing Model")
#------------------------- EARLY STOPPING -------------------------
    if average_val_loss >= last_epoch_loss:
        worse_loss_counter +=1
    else:
        worse_loss_counter = 0
    if worse_loss_counter >= 5:
        print("Stopped training early due to the model not getting better validation loss")
        break
    last_epoch_loss = average_val_loss

lab1_1.end()

Epoch [1/5] train loss: 0.693, val loss: 0.692, lr: 0.000100
Saved The Best Performing Model
Epoch [2/5] train loss: 0.689, val loss: 0.690, lr: 0.000100
Saved The Best Performing Model


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : immense_courtyard_5963
COMET INFO:     url                   : https://www.comet.com/kanskejoanna/lab1/d0244f827e514a1d81733e66b2ce98f9
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     model 1.1.1 - learning rate       : 0.0001
COMET INFO:     model 1.1.1 - train loss [5]      : (0.6704978860657791, 0.6926964685834688)
COMET INFO:     model 1.1.1 - validation loss [5] : (0.6767413765192032, 0.6917894929647446)
COMET INFO:   Parameters:
COMET INFO:     learning_rate : 0.0001
COMET INFO:     steps         : 5
COMET INFO:   Uploads:
COMET INFO:     environment details      : 1
COMET INFO:     filename                 : 1
COMET INFO:     git met

Epoch [3/5] train loss: 0.685, val loss: 0.687, lr: 0.000100
Saved The Best Performing Model
Epoch [4/5] train loss: 0.679, val loss: 0.683, lr: 0.000100
Saved The Best Performing Model
Epoch [5/5] train loss: 0.670, val loss: 0.677, lr: 0.000100
Saved The Best Performing Model


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.


## 1.1 ANN: Test run
The best model is tested

In [7]:
#Test the model
model_1_1.load_state_dict(torch.load("BestModel_1_1.pth"))
model_1_1.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in val_loader_LSTM:
        output = model_1_1(batch_x)
        val_loss = criterion(output, batch_y)
        test_loss += val_loss.item()

        HighestValue, predicted = torch.max(output, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

average_test_loss = test_loss / len(val_loader_LSTM)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

Test loss: 0.677
Test accuracy: 79.00%


C:\Users\david\AppData\Local\Temp\ipykernel_46684\2998430035.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_1_1.load_state_dict(torch.load("BestModel_1_1.pth"))


# Task 1.1 LSTM network
A neural network based on LSTM or bidirectional LSTM (Bi-LSTM) layers.

In [8]:
# settings
epochs = epochs #50
lr=0.0001

# scheduler settings
patience = 3
factor = 0.1

# initialize comet
lab1_1_2 = comet_ml.Experiment(
    api_key="wCXnRD5xewUGxCxBYe8ePt4JY",
    workspace="kanskejoanna",
    project_name="lab1",
    name="model 1.1.2 - Bi-LSTM",
)

# Report multiple hyperparameters using a dictionary:
hyper_params = {
   "learning_rate": lr,
   "steps": epochs
}
lab1_1_2.log_parameters(hyper_params)

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kanskejoanna/lab1/7dc9086fbb06450a813d2bae113e0a66

COMET WARNING: Unknown error exporting current conda environment
COMET WARNING: Unknown error retrieving Conda package as an explicit file
COMET WARNING: Unknown error retrieving Conda information


## 1.1 LSTM: Training and Validation
The model is trained, validated and the best model is saved.

In [ ]:

model_LSTM = LSTM()
criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.SGD(model_LSTM.parameters(), lr=0.0001)
optimizer = torch.optim.Adam(model_LSTM.parameters(), lr=lr)

# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#         optimizer,
#         mode="min",
#         factor = 0.1,
#         patience = 3 # Wait x amount of epochs before reducing lr
#     )

best_val_loss_LSTM = float('inf')
last_epoch_loss_LSTM = 0
worse_loss_counter_LSTM = 0

for epoch in range(epochs):
    model_LSTM.train()

    running_train_loss_LSTM = 0.0
    for batch_x, batch_y in train_loader_LSTM:
#------------------------- TRAINING -------------------------
        optimizer.zero_grad()
        output = model_LSTM(batch_x)
        loss_LSTM = criterion(output, batch_y)
        loss_LSTM.backward()
        optimizer.step()
        running_train_loss_LSTM += loss_LSTM.item()
    average_train_loss_LSTM = running_train_loss_LSTM/len(train_loader_LSTM)
    lab1_1_2.log_metric("model 1.1.2 - train loss", average_train_loss_LSTM, step=epoch)

#------------------------- VALIDATION -------------------------
    model_LSTM.eval()
    val_running_loss_LSTM = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader_LSTM:
            output = model_LSTM(batch_x)
            val_loss_LSTM = criterion(output, batch_y)
            val_running_loss_LSTM += val_loss_LSTM.item()
        average_val_loss_LSTM = val_running_loss_LSTM/len(val_loader_LSTM)
        lab1_1_2.log_metric("model 1.1.2 - validation loss", average_val_loss_LSTM, step=epoch)

    # scheduler.step(average_val_loss_LSTM)
    lab1_1_2.log_metric("model 1.1.2 - learning rate", optimizer.param_groups[0]['lr'], step=epoch)

#------------------------- VISUALIZATION -------------------------
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"train loss: {average_train_loss_LSTM:.3f}, "
        f"val loss: {average_val_loss_LSTM:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )
#------------------------- SAVING BEST MODEL -------------------------
    if average_val_loss_LSTM < best_val_loss_LSTM:
        best_val_loss_LSTM = average_val_loss_LSTM
        torch.save(model_LSTM.state_dict(), "BestModel_LSTM.pth")
        print("Saved The Best Performing Model")
#------------------------- EARLY STOPPING -------------------------
    if average_val_loss_LSTM >= last_epoch_loss_LSTM:
        worse_loss_counter_LSTM +=1
    else:
        worse_loss_counter_LSTM = 0
    if worse_loss_counter_LSTM >= 5:
        print("Stopped training early due to the model not getting better validation loss")
        break
    last_epoch_loss_LSTM = average_val_loss_LSTM



lab1_1_2.end()

Epoch [1/5] train loss: 0.692, val loss: 0.696, lr: 0.000100
Saved The Best Performing Model
Epoch [2/5] train loss: 0.690, val loss: 0.695, lr: 0.000100
Saved The Best Performing Model
Epoch [3/5] train loss: 0.691, val loss: 0.694, lr: 0.000100
Saved The Best Performing Model
Epoch [4/5] train loss: 0.688, val loss: 0.692, lr: 0.000100
Saved The Best Performing Model


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : unexpected_jigsaw_8460
COMET INFO:     url                   : https://www.comet.com/kanskejoanna/lab1/7dc9086fbb06450a813d2bae113e0a66
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     model 1.1.2 - learning rate       : 0.0001
COMET INFO:     model 1.1.2 - train loss          : 0.6704978860657791
COMET INFO:     model 1.1.2 - validation loss [5] : (0.690672367811203, 0.6962715834379196)
COMET INFO:   Parameters:
COMET INFO:     learning_rate : 0.0001
COMET INFO:     steps         : 5
COMET INFO:   Uploads:
COMET INFO:     environment details      : 1
COMET INFO:     filename                 : 1
COMET INFO:     git metadata             : 1
C

Epoch [5/5] train loss: 0.685, val loss: 0.691, lr: 0.000100
Saved The Best Performing Model


COMET INFO: Please wait for metadata to finish uploading (timeout is 3600 seconds)
COMET INFO: Uploading 1 metrics, params and output messages


## 1.1 LSTM Test run
The best model is tested

In [10]:
model_LSTM.load_state_dict(torch.load("BestModel_LSTM.pth"))
model_LSTM.eval()

test_loss_LSTM = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in val_loader_LSTM:
        output = model_LSTM(batch_x)
        val_loss_LSTM = criterion(output, batch_y)
        test_loss_LSTM += val_loss_LSTM.item()

        HighestValue, predicted = torch.max(output, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

average_test_loss_LSTM = test_loss_LSTM / len(val_loader_LSTM)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss_LSTM:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")



Test loss: 0.691
Test accuracy: 47.00%


C:\Users\david\AppData\Local\Temp\ipykernel_46684\3094863469.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_LSTM.load_state_dict(torch.load("BestModel_LSTM.pth"))

# Task 1.2: Transformers Implementation
For this task, you will implement your transformer in PyTorch. 

In [11]:
# settings
epochs = epochs #50
best_val_loss_bert = float('inf')
dataset = "amazon_cells_labelled_LARGE_25K.txt"
#lr = 0.0001

# scheduler settings
patience = 5
factor = 0.1
patience_counter = 0

# initialize comet
lab1_2 = comet_ml.Experiment(
    api_key="wCXnRD5xewUGxCxBYe8ePt4JY",
    workspace="kanskejoanna",
    project_name="lab1",
    display_name="model 1.1.2 - Bi-LSTM",
)

# Report multiple hyperparameters using a dictionary:
hyper_params = {
   "learning_rate": lr,
   "steps": epochs
}
lab1_2.log_parameters(hyper_params)

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kanskejoanna/lab1/cbd65d73ac6248ff90f33a1cf3c485b0

COMET WARNING: Unknown error exporting current conda environment
COMET WARNING: Unknown error retrieving Conda package as an explicit file
COMET WARNING: Unknown error retrieving Conda information


In [12]:
df = pd.read_csv(dataset, delimiter='\t', header=None)
sentiment_mapping = {0: 'Negative', 1: 'Positive'}
df['Rating'] = df[1].map(sentiment_mapping)#df['Rating'] = df[1].map(sentiment_mapping)
print('Head: \n')
print(df.head())

print('Rating: \n')
print(df['Rating'].value_counts())

Head: 

                                                   0  1    Rating
0  I've read this book with much expectation, it ...  0  Negative
1  love it...very touch.it's to bad that in the d...  1  Positive
2  The creepiest book I've ever read! It's a cree...  1  Positive
3  It starts off a bit slow, but once the product...  1  Positive
4  As good as this book may be, the print quality...  0  Negative
Rating: 

Rating
Positive    15116
Negative     9884
Name: count, dtype: int64


In [5]:
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, csv_file, tokenizer, max_length):
        self.dataset = pd.read_csv(csv_file, delimiter='\t', header=None, names=['Sentence', 'Class'])
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label_dict = {0: 'Negative', 1: 'Positive'}
    

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        review = self.dataset.iloc[idx, 0]
        sentiment = self.dataset.iloc[idx, 1]
        label = self.label_dict[sentiment]

        encoding = self.tokenizer(
            review,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'review_text': review,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(sentiment, dtype=torch.long)
        }

In [14]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
review_dataset = ReviewDataset(dataset, tokenizer, 512)
review_dataset[0]
tokenizer.decode(review_dataset[0]['input_ids'])

"[CLS] i ' ve read this book with much expectation, it was very boring all through out the book [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD

In [15]:
train_size_bert = int(0.7 * len(df))
val_size_bert = int(0.15 * len(df))
test_size_bert = int(0.15 * len(df))
train_dataset_bert, val_dataset_bert, test_dataset_bert = random_split(review_dataset, [train_size_bert, val_size_bert, test_size_bert])

train_loader_bert = DataLoader(train_dataset_bert, batch_size=16, shuffle=True)
val_loader_bert = DataLoader(val_dataset_bert, batch_size=16, shuffle=False)
test_loader_bert = DataLoader(test_dataset_bert, batch_size=16, shuffle=False)

print(f"Number of training samples: {len(train_dataset_bert)}")
print(f"Number of validation samples: {len(val_dataset_bert)}")
print(f"Number of test samples: {len(test_dataset_bert)}")

Number of training samples: 17500
Number of validation samples: 3750
Number of test samples: 3750


In [16]:
class CustomDistilBertForClassification(nn.Module):
    def __init__(self, num_labels=2):
        super(CustomDistilBertForClassification, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.pre_classifier = nn.Linear(self.distilbert.config.dim, 64)
        self.dropout = nn.Dropout(0.4)
        self.classifier = nn.Linear(64, num_labels)

    def forward(self, input_ids, attention_mask):
        distilbert_output = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = distilbert_output[0]  # (batch_size, sequence_length, hidden_size)
        pooled_output = hidden_state[:, 0]  # Take the [CLS] token representation
        pooled_output = self.dropout(pooled_output)
        pooled_output = self.pre_classifier(pooled_output)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

In [17]:
model = CustomDistilBertForClassification()

print(model.distilbert)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

## 1.2 Transformer: Training and Validation

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
best_val_loss_bert = float("inf")
patience_counter = 0
unfreeze_epoch = 10

# Freeze DistilBERT layers initially
for param in model.distilbert.parameters():
    param.requires_grad = False

def trainable(p):
    return p.requires_grad

# Add weight decay (L2 regularization) to reduce overfitting
optimizer = torch.optim.Adam(filter(trainable, model.parameters()), lr=5e-5)

model.train()
for epoch in range(epochs):
    # Unfreeze DistilBERT layers at epoch 10 and initiate fine-tuning
    if epoch == unfreeze_epoch:
        print("Unfreezing DistilBERT layers... Initiating fine-tuning!")
        
        for param in model.distilbert.parameters():
            param.requires_grad = True
        
        # Recreate optimizer with all parameters and lower learning rate
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=1e-5,
            weight_decay=0.01
        )
    
    # Training phase
    total_loss = 0
    for batch in train_loader_bert:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader_bert)
    lab1_2.log_metric("model 1.2 - train loss", avg_train_loss, step=epoch)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader_bert:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader_bert)
    lab1_2.log_metric("model 1.2 - validation loss", avg_val_loss, step=epoch)

    lab1_2.log_metric("model 1.2 - learning rate", optimizer.param_groups[0]['lr'], step=epoch)
    
    # Print metrics
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
    
    # Save best model and implement early stopping
    if avg_val_loss < best_val_loss_bert:
        best_val_loss_bert = avg_val_loss
        torch.save(model.state_dict(), "BestModel_DistilBert.pth")
        print(f"Saved best model with validation loss: {avg_val_loss:.4f}")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break

lab1_2.end()

Epoch 1/5, Train Loss: 0.3548, Val Loss: 0.3399
Saved best model with validation loss: 0.3399
Epoch 2/5, Train Loss: 0.3457, Val Loss: 0.3207
Saved best model with validation loss: 0.3207
Epoch 3/5, Train Loss: 0.3227, Val Loss: 0.3113
Saved best model with validation loss: 0.3113
Epoch 4/5, Train Loss: 0.2900, Val Loss: 0.3430


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : maroon_gauge_5253
COMET INFO:     url                   : https://www.comet.com/kanskejoanna/lab1/cbd65d73ac6248ff90f33a1cf3c485b0
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     model 1.2 - learning rate       : 5e-05
COMET INFO:     model 1.2 - validation loss [5] : (0.3113142360715156, 0.3430045192546033)
COMET INFO:   Parameters:
COMET INFO:     learning_rate : 0.0001
COMET INFO:     steps         : 5
COMET INFO:   Uploads:
COMET INFO:     environment details      : 1
COMET INFO:     filename                 : 1
COMET INFO:     git metadata             : 1
COMET INFO:     git-patch (uncompressed) : 1 (46.21 KB)
COMET INFO:     installed

Epoch 5/5, Train Loss: 0.2596, Val Loss: 0.3139


## 1.2 Transformer: Testing

In [19]:

model.load_state_dict(torch.load("BestModel_DistilBert.pth"))
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader_bert:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        test_loss += loss.item()
        
        # Calculate accuracy
        _, predicted = torch.max(logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

avg_test_loss = test_loss / len(val_loader_bert)
test_accuracy = 100 * (correct / total)

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"Correct Predictions: {correct}/{total}")

C:\Users\david\AppData\Local\Temp\ipykernel_46684\116359246.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("BestModel_DistilBert.pth"))

Test Loss: 0.3042
Test Accuracy: 87.39%
Correct Predictions: 3277/3750


## 1.2 GPT 2

In [66]:
class CustomGPT2(nn.Module):
    def __init__(self, num_labels=2):
        super().__init__()

        self.gpt2 = GPT2Model.from_pretrained("distilgpt2")
        hidden_size = self.gpt2.config.hidden_size

        self.dropout = nn.Dropout(0.1)
        self.fc1 = nn.Linear(hidden_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.relu = nn.ReLU()
        self.classifier = nn.Linear(64, num_labels)
        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask):
        # input_ids is tokenized text
        # attention mask is padding for each sentence
        outputs = self.gpt2(input_ids=input_ids, attention_mask=attention_mask)

        hidden_states = outputs.last_hidden_state # A vector representation for each token in the sequence, taken from the final layer of GPT-2

        #last_token_idx = attention_mask.sum(dim=1) - 1 # Finds the last word in the sentence
        #batch_size = input_ids.size(0) # How much sentences in each batch
        mask = attention_mask.unsqueeze(-1).float()   # (batch_size, seq_len, 1)
        masked_hidden = hidden_states * mask

        summed = masked_hidden.sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1e-9)
        pooled_output = summed / lengths

    
        #pooled_output = hidden_states[ # Take the last token for each sentence
        #    torch.arange(batch_size, device=input_ids.device),
        #    last_token_idx
        #]

        x = self.dropout(pooled_output)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)

        logits = self.classifier(x)
        return logits

In [67]:

from transformers import GPT2Tokenizer, GPT2Model

tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

#model = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=2)
model = CustomGPT2(num_labels=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

c:\Users\david\anaconda3\envs\tretolv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\david\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

cuda


In [68]:
csv_file_path = "amazon_cells_labelled_LARGE_25K.txt"
dataset = ReviewDataset(csv_file_path, tokenizer, max_length=128)

print(dataset[0])
print(tokenizer.decode(dataset[0]["input_ids"], skip_special_tokens=True))

{'review_text': "I've read this book with much expectation, it was very boring all through out the book", 'input_ids': tensor([   40,  1053,  1100,   428,  1492,   351,   881, 17507,    11,   340,
          373,   845, 14262,   477,   832,   503,   262,  1492, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 5

In [ ]:
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size]
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Train: {len(train_dataset)}")
print(f"Val:   {len(val_dataset)}")
print(f"Test:  {len(test_dataset)}")

Train: 17500
Val:   3750
Test:  3750


In [70]:
gpt2_logger = comet_ml.Experiment(
    api_key="wCXnRD5xewUGxCxBYe8ePt4JY",
    workspace="kanskejoanna",
    project_name="lab1",
    display_name="GPT2 Model",
)

hyper_params = {
   "learning_rate": lr,
   "steps": epochs
}
gpt2_logger.log_parameters(hyper_params)

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kanskejoanna/lab1/37ba1554f44c44a3ba93fad0fe4a6a92

COMET WARNING: Unknown error exporting current conda environment
COMET WARNING: Unknown error retrieving Conda package as an explicit file
COMET WARNING: Unknown error retrieving Conda information


In [74]:
criterion = nn.CrossEntropyLoss()
best_val_loss = float("inf")
patience_counter = 0
patience = 14
unfreeze_epoch = 10

for param in model.gpt2.parameters():
    param.requires_grad = False

def trainable(p):
    return p.requires_grad

optimizer = torch.optim.AdamW(filter(trainable, model.parameters()), lr=3e-3, weight_decay=0.03) # Now we only have backpropagation on trainable parameters
# Gives optimizern only parameters where reguires_grad = true

# Learning rate scheduler to reduce LR when validation loss plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    min_lr=1e-7
)

for epoch in range(epochs):
    # Unfreeze
    if epoch == unfreeze_epoch:
        print("Unfreezing GPT-2 Layers... Initiating Fine-Tuning!")
        for param in model.gpt2.parameters():
            param.requires_grad = True

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=2e-5,
            weight_decay=0.01
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=8,
            min_lr=1e-7
        )
        
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)
    gpt2_logger.log_metric("GPT2 - train loss", avg_train_loss, step=epoch)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    gpt2_logger.log_metric("GPT2 - validation loss", avg_val_loss, step=epoch)
    
    # Step scheduler
    scheduler.step(avg_val_loss)
    gpt2_logger.log_metric("GPT2 - Learning Rate", optimizer.param_groups[0]['lr'], step=epoch)
    
    # Print metrics
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
    
    # Save best model and implement early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "BestModel_GPT2.pth")
        print(f"Saved best model with validation loss: {avg_val_loss:.4f}")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break

gpt2_logger.end()

Epoch 1/1000, Train Loss: 0.6698, Val Loss: 0.6698
Saved best model with validation loss: 0.6698
Epoch 2/1000, Train Loss: 0.6699, Val Loss: 0.6693
Saved best model with validation loss: 0.6693
Epoch 3/1000, Train Loss: 0.6699, Val Loss: 0.6694
Epoch 4/1000, Train Loss: 0.6698, Val Loss: 0.6693
Saved best model with validation loss: 0.6693
Epoch 5/1000, Train Loss: 0.6698, Val Loss: 0.6693
Epoch 6/1000, Train Loss: 0.6699, Val Loss: 0.6693
Epoch 7/1000, Train Loss: 0.6697, Val Loss: 0.6694
Epoch 8/1000, Train Loss: 0.6699, Val Loss: 0.6693
Epoch 9/1000, Train Loss: 0.6697, Val Loss: 0.6694
Epoch 10/1000, Train Loss: 0.6697, Val Loss: 0.6694
Unfreezing GPT-2 Layers... Initiating Fine-Tuning!
Epoch 11/1000, Train Loss: 0.6697, Val Loss: 0.6694
Epoch 12/1000, Train Loss: 0.6697, Val Loss: 0.6694
Epoch 13/1000, Train Loss: 0.6697, Val Loss: 0.6694
Epoch 14/1000, Train Loss: 0.6697, Val Loss: 0.6694
Epoch 15/1000, Train Loss: 0.6696, Val Loss: 0.6694
Epoch 16/1000, Train Loss: 0.6697, Val L

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : straight_car_2635
COMET INFO:     url                   : https://www.comet.com/kanskejoanna/lab1/37ba1554f44c44a3ba93fad0fe4a6a92
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     GPT2 - Learning Rate [24]   : (2e-05, 0.003)
COMET INFO:     GPT2 - train loss [24]      : (0.669633567578396, 0.671653401699101)
COMET INFO:     GPT2 - validation loss [24] : (0.6693313786324034, 0.6698204339818752)
COMET INFO:   Parameters:
COMET INFO:     learning_rate : 0.0001
COMET INFO:     steps         : 1000
COMET INFO:   Uploads:
COMET INFO:     environment details      : 1
COMET INFO:     filename                 : 1
COMET INFO:     git metadata         

Epoch 18/1000, Train Loss: 0.6697, Val Loss: 0.6693
Early stopping triggered after 18 epochs


COMET INFO: Uploading 31 metrics, params and output messages
COMET INFO: Please wait for assets to finish uploading (timeout is 10800 seconds)
COMET INFO: All assets have been sent, waiting for delivery confirmation


### Test GPT2

In [75]:

model.load_state_dict(torch.load("BestModel_GPT2.pth"))
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        test_loss += loss.item()
        
        # Calculate accuracy
        _, predicted = torch.max(logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

avg_test_loss = test_loss / len(val_loader_bert)
test_accuracy = 100 * (correct / total)

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"Correct Predictions: {correct}/{total}")

C:\Users\david\AppData\Local\Temp\ipykernel_46684\3752993064.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("BestModel_GPT2.pth"))


Test Loss: 0.6802
Test Accuracy: 58.48%
Correct Predictions: 2193/3750


## Roberta

In [10]:
model_name = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [11]:
csv_file_path = "amazon_cells_labelled_LARGE_25K.txt"
dataset = ReviewDataset(csv_file_path, tokenizer, max_length=128)

print(dataset[0])
print(tokenizer.decode(dataset[0]["input_ids"], skip_special_tokens=True))

{'review_text': "I've read this book with much expectation, it was very boring all through out the book", 'input_ids': tensor([    0,   100,   348,  1166,    42,  1040,    19,   203,  9250,     6,
           24,    21,   182, 15305,    70,   149,    66,     5,  1040,     2,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,  

In [12]:
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size]
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Train: {len(train_dataset)}")
print(f"Val:   {len(val_dataset)}")
print(f"Test:  {len(test_dataset)}")

Train: 17500
Val:   3750
Test:  3750


In [13]:
roberta_logger = comet_ml.Experiment(
    api_key="wCXnRD5xewUGxCxBYe8ePt4JY",
    workspace="kanskejoanna",
    project_name="lab1",
    display_name="Roberta-base Model",
)

lr = 5e-4
epochs = 1000
hyper_params = {
   "learning_rate": lr,
   "steps": epochs
}
roberta_logger.log_parameters(hyper_params)

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, sklearn.
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : federal_chalet_3974
COMET INFO:     url                   : https://www.comet.com/kanskejoanna/lab1/195f4487db424981ab2010d5d9e8e970
COMET INFO:   Uploads:
COMET INFO:     environment details : 1
COMET INFO:     filename            : 1
COMET INFO:     git metadata        : 1
COMET INFO:     installed packages  : 1
COMET INFO:     notebook            : 1
COMET INFO:     source_code         : 1
COMET INFO: 
COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, sklearn.
COMET WARNING: As y

In [14]:
class CustomRoberta(nn.Module):
    def __init__(self, num_labels=2):
        super().__init__()

        self.roberta = AutoTokenizer.from_pretrained("roberta-base")
        hidden_size = self.roberta.config.hidden_size

        self.dropout = nn.Dropout(0.1)
        self.fc1 = nn.Linear(hidden_size, 128)
        self.fc2 = nn.Linear(128,64)
        self.classifier = nn.Linear(64, num_labels)
        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask):
        roberta_output = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = roberta_output[0]  # (batch_size, sequence_length, hidden_size)
        pooled_output = hidden_state[:, 0]  # Take the [CLS] token representation
        pooled_output = self.dropout(pooled_output)
        pooled_output = self.pre_classifier(pooled_output)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits


In [16]:
criterion = nn.CrossEntropyLoss()
best_val_loss = float("inf")
patience_counter = 0
patience = 10
unfreeze_epoch = 10


for param in model.roberta.parameters():
    param.requires_grad = False

def trainable(p):
    return p.requires_grad

optimizer = torch.optim.AdamW(
    filter(trainable, model.parameters()),
    lr=5e-4,
    weight_decay=0.03
)

#Just for the model to work, the scheduler has a patience of 100 epochs. What's interesting is the scheduler used after the Unfreezing of the RoBERTa layers
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=100,
            min_lr=1e-7
        )

for epoch in range(epochs):
    if epoch == unfreeze_epoch:
        print(f"Unfreezing RoBERTa layers after {unfreeze_epoch} epochs... Initiating fine-tuning!")

        for param in model.roberta.parameters():
            param.requires_grad = True

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=2e-5,
            weight_decay=0.03
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=8,
            min_lr=1e-7
        )

    model.train()
    total_loss = 0.0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device) # Retrieve the token ID's and move them to the model's device
        attention_mask = batch['attention_mask'].to(device) # Retrieve attention mask from input batch and send it to the same device as the model
        labels = batch['labels'].to(device) # Same but for the labels

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask) # Use attention mask so padding tokens do not affect the attention computation
        logits = outputs.logits
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    roberta_logger.log_metric("RoBERTa - train loss", avg_train_loss, step=epoch)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask) # needed 
            logits = outputs.logits
            loss = criterion(logits, labels)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)
    roberta_logger.log_metric("RoBERTa - validation loss", avg_val_loss, step=epoch)

    scheduler.step(avg_val_loss)
    roberta_logger.log_metric("RoBERTa - learning rate", optimizer.param_groups[0]['lr'], step=epoch)

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "BestModel_RoBERTa.pth")
        print(f"Saved best model with validation loss: {avg_val_loss:.4f}")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break

roberta_logger.end()

Epoch 1/1000, Train Loss: 0.3679, Val Loss: 0.3328
Saved best model with validation loss: 0.3328
Epoch 2/1000, Train Loss: 0.3615, Val Loss: 0.2682
Saved best model with validation loss: 0.2682
Epoch 3/1000, Train Loss: 0.3536, Val Loss: 0.2574
Saved best model with validation loss: 0.2574
Epoch 4/1000, Train Loss: 0.3392, Val Loss: 0.2529
Saved best model with validation loss: 0.2529
Epoch 5/1000, Train Loss: 0.3465, Val Loss: 0.2576
Epoch 6/1000, Train Loss: 0.3389, Val Loss: 0.2744
Epoch 7/1000, Train Loss: 0.3372, Val Loss: 0.2513
Saved best model with validation loss: 0.2513
Epoch 8/1000, Train Loss: 0.3318, Val Loss: 0.2541
Epoch 9/1000, Train Loss: 0.3348, Val Loss: 0.2576
Epoch 10/1000, Train Loss: 0.3318, Val Loss: 0.2497
Saved best model with validation loss: 0.2497
Unfreezing RoBERTa layers after 10 epochs... Initiating fine-tuning!
Epoch 11/1000, Train Loss: 0.2342, Val Loss: 0.1899
Saved best model with validation loss: 0.1899
Epoch 12/1000, Train Loss: 0.1574, Val Loss: 0

COMET ERROR: Failed to send parameters batch message, got 502 b'<html>\r\n<head><title>502 Bad Gateway</title></head>\r\n<body>\r\n<center><h1>502 Bad Gateway</h1></center>\r\n<hr><center>nginx</center>\r\n</body>\r\n</html>\r\n'
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : dry_canid_708
COMET INFO:     url                   : https://www.comet.com/kanskejoanna/lab1/15c86444df464e5fa51970893bd20ec2
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     RoBERTa - learning rate [23]   : (1e-05, 0.0005)
COMET INFO:     RoBERTa - train loss [24]      : (0.00939588211527839, 0.4482843997518803)
COMET INFO:     RoBERTa - validation loss [24] : (0.17880883296119407, 0.5572833368055167)
COMET INFO: 

Epoch 23/1000, Train Loss: 0.0094, Val Loss: 0.5573
Early stopping triggered after 23 epochs


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, sklearn.


In [18]:
model.load_state_dict(torch.load("BestModel_RoBERTa.pth"))
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        loss = criterion(logits, labels)
        test_loss += loss.item()
        
        # Calculate accuracy
        _, predicted = torch.max(logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

avg_test_loss = test_loss / len(val_loader)
test_accuracy = 100 * (correct / total)

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"Correct Predictions: {correct}/{total}")

C:\Users\david\AppData\Local\Temp\ipykernel_48104\3521924318.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("BestModel_RoBERTa.pth"))


Test Loss: 0.1747
Test Accuracy: 93.65%
Correct Predictions: 3512/3750


## Task 1.3 Comparison
Here, you should compare of all three models; you are requested to use the same test dataset
for Simple ANN, LSTM based model and the Transformer to answer the following:

• Compare the performance of the two models and explain in which scenarios you would
prefer one over the other.

• How did the two models’ complexity, accuracy, and efficiency differ? Did one model
outperform the other in specific scenarios or tasks? If so, why?

• What insights did you obtain concerning data amount to train? Embedding utilized?
Architectural choices made?